In [3]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

An LLM is a function call that takes a prompt and returns a response

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [5]:
llm("Hey, what FIFA world cup matches are playing today")

'I can help, but I don’t have live match schedules built in, so I can’t reliably tell you which FIFA World Cup matches are on **today** without a date or current schedule source.\n\nIf you want, I can still help in either of these ways:\n\n1. **Tell me the date and time zone** you mean, and I can help you figure out what would be happening on that day.\n2. **Paste the tournament schedule** or a screenshot/link, and I’ll extract today’s matches for you.\n3. If you want, I can also tell you **how to quickly check today’s World Cup fixtures** on FIFA’s official site or major sports apps.\n\nIf you want the fastest route, send me:\n- the **date**\n- your **time zone**\n- and whether you mean **men’s or women’s World Cup**\n\nand I’ll help from there.'

In [6]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—usually you can still join now, but it depends on the course’s enrollment rules and whether there’s still a spot or the course is still open.

A few possibilities:
- **Open enrollment:** you can join immediately.
- **Limited enrollment / capped class:** you can join only if there’s room.
- **Started already:** you may still be able to join late, but you might need instructor approval or to catch up on missed material.

If you want, I can help you draft a quick message to the course organizer/instructor asking to join.


Sending a question to the prompt will return an answer based on the LLM's knowledge base and returns a generic answer

In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

Adding context(string in this case but can be any type of source i reckon), provides the LLM a more specific knowledge base that can return a more useful answer

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.


A prompt can contain different sections (prompt, question, context)

Prompt - High level instructions

Question - The actual question from the user

Context - A specific knowledge base the LLM reads first before answering

RAGs are Retrieval Augmented Generation

Generation - The output of an LLM

Retrieval Augmented - the Context we provide and allow the LLM to read



In [10]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

A RAG consists of 3 components:

- Search : Looks into context to find the relevant results
- Prompt : Combines Search Results and Question
- LLM : Takes the prompt and yields a result

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [14]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

Search is the first component of a RAG. Before knowledge can be searched, it needs to be indexed. The index comes from a lightweight library called minsearch that uses text matching.

Text Fields is what gets indexed

Keyword Fields works like a prefilter to reduce the search space

In [19]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

When running a search, you can boost/filter/limit results of search

Boost when you know some text fields are more important than others in organizing search results

In [20]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [27]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [28]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [29]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
search_results = search(question)

In [31]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [32]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

Prompt building consist of 2 parts:
- The instructions: High level system instructions for the prompt (never changes)
- The user prompt: User input (always changes) with context from search

In [33]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [34]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [38]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [39]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [41]:
response

Response(id='resp_0a1a9fa050a9acf7006a3222fbe1d48198bb208fe16aaec1d5', created_at=1781670651.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0a1a9fa050a9acf7006a3222fca0a8819886fafbc969074901', content=[ResponseOutputText(annotations=[], text='Yes — you can join now and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, since certificates are only available during the live cohort.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781670653.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(

In [42]:
type(response)

openai.types.responses.response.Response

In [43]:
response.output[0]

ResponseOutputMessage(id='msg_0a1a9fa050a9acf7006a3222fca0a8819886fafbc969074901', content=[ResponseOutputText(annotations=[], text='Yes — you can join now and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, since certificates are only available during the live cohort.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [44]:
response.output[0].content[0].text

'Yes — you can join now and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, since certificates are only available during the live cohort.'

In [45]:
response.output_text

'Yes — you can join now and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open, since certificates are only available during the live cohort.'

In [46]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=42, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=522)

In [47]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000549

In [48]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [49]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [50]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join and start learning now. If you want to receive a certificate, though, you need to submit your project while submissions are still open.


In [53]:
answer = rag("What model am i using?")
print(answer)

I don't know.


In [54]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort** and pass the **Capstone project**.\n\nNotes:\n- **Self-paced mode does not qualify** for a certificate.\n- **Homework is not mandatory** for the certificate.\n- Your **official name** in your course profile will appear on the certificate.'

Takeaways:
To build a RAG I need:
1. A question
2. A queryable knowledge base
3. An LLM to process my question with the right context

Components:
1. Searchable context requires:
   - Relevant information
   - Index
   - Scoring function
3. Prompt Function
   - Instructions
   - Question
   - Context(processed context)
4. LLM
   - Processing
   - Pricing
